# Train YOLOv10n / YOLO11n / YOLO12n / YOLOX-s on MOT17 (train_half → val_half)

So sánh công bằng 4 detector theo bài báo *"Performance Comparison of YOLOv10, YOLOv11, and YOLOv12 Models on Human Detection Datasets"* (Brilliance, 2025).

| | YOLOv10n / YOLO11n / YOLO12n | YOLOX-s |
|---|---|---|
| Framework | Ultralytics | ByteTrack/YOLOX (`tools/train.py`) |
| Khởi tạo | COCO pretrained (`*.pt`) | COCO pretrained (`yolox_s.pth`) |
| Input (mặc định của model) | 640 | 608×1088 (mặc định ByteTrack MOT) |
| Train / Val | MOT17 `train_half` / `val_half` (FRCNN, 7 video) | giống |
| Tham số (Bảng 2 của bài báo) | SGD, lr0 0.01, lrf 0.1, momentum 0.937, batch 16, 100 epoch, warmup 10, mosaic 0.5, close_mosaic 20, cos LR | giống (`exps/example/mot/yolox_s_mot17_half.py`) |

**Chuẩn bị trên Kaggle**
1. Accelerator: **GPU T4 x2** (YOLOX sẽ train song song trên GPU thứ 2). Bật **Internet**.
2. *Add Input* → một dataset MOT17 có thư mục `train/MOT17-XX-FRCNN/{img1,gt,det}` (notebook tự tìm đường dẫn).
3. Repo trên GitHub phải có các thay đổi mới nhất (`exps/example/mot/yolox_s_mot17_half.py`, `exps/example/mot/yolox_tiny_mot17_half.py`, `tools/track_dma.py`, `yolox/evaluators/mot_evaluator.py`).

**Lưu checkpoint khi hết session.** `/kaggle/working` bị xoá khi session kết thúc, nên notebook tự sao lưu checkpoint (`last`/`best` của cả 4 model) lên một **Kaggle Dataset riêng tư** sau mỗi `BACKUP_EVERY_MIN` phút và sau mỗi model.
- Lần đầu: vào *Add-ons → Secrets*, thêm `KAGGLE_USERNAME` và `KAGGLE_KEY` (lấy từ kaggle.com → Settings → API → *Create New Token*), rồi bật cả hai cho notebook này.
- Khi session bị ngắt: mở session mới, *Add Input* → dataset `mot17-half-4models-ckpt` (tab *Your Work → Datasets*), rồi **Run All**. Notebook tự khôi phục và **train tiếp từ epoch đang dở**. Model nào xong rồi sẽ được bỏ qua.

## 1. Cấu hình

In [1]:
import os, glob, json, shutil, subprocess, sys, time
from collections import defaultdict

REPO_URL = "https://github.com/khangkaka066/ByteTrack-DMA-LTC-Motion-Tracker.git"
REPO_DIR = "/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker"
MOT17_TRAIN_SRC = "/kaggle/input/datasets/hongchtrung/mot17-train/train"  # None = tự tìm trong /kaggle/input; hoặc ghi đường dẫn tới thư mục train/
YOLO_DATA_DIR = "/kaggle/working/mot17_half_yolo"
RUNS_DIR = "/kaggle/working/runs"

# Chọn model cần train trong session này
TRAIN_YOLOX = True
ULTRA_MODELS = ["yolov10s.pt", "yolo11s.pt", "yolo12s.pt", "yolo26s.pt"]

# Bảng 2 của bài báo (dùng chung cho cả 4 model)
HP = dict(
    epochs=100, batch=16, optimizer="SGD", lr0=0.01, lrf=0.1, momentum=0.937,
    weight_decay=5e-4, warmup_epochs=10, mosaic=0.5, close_mosaic=20, cos_lr=True,
)
ULTRA_IMGSZ = 640               # mặc định Ultralytics
YOLOX_IMGSZ = (608, 1088)       # (cao, rộng), bội số của 32. Mặc định ByteTrack MOT; vd (640, 640) để giống Ultralytics
SEED = 0
WORKERS = 4                     # Kaggle có 4 vCPU

# Bảng 2 còn ghi IoU 0.6 / conf 0.25: đó là ngưỡng lúc predict. Không đưa vào train,
# vì Ultralytics sẽ dùng conf 0.25 khi validate mỗi epoch -> mAP bị thấp và best.pt chọn sai.
EVAL_CONF = 0.001               # giống mặc định khi Ultralytics validate
EVAL_NMS = 0.7

# Sao lưu checkpoint lên Kaggle Dataset riêng tư (cần secrets KAGGLE_USERNAME / KAGGLE_KEY)
BACKUP_ENABLED = True
BACKUP_SLUG = "mot17-half-4models-ckpt"
BACKUP_EVERY_MIN = 30

## 2. Clone repo và cài thư viện

In [2]:
%cd /kaggle/working
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
%cd {REPO_DIR}
!pip install -q -r requirements-kaggle.txt
!pip install -q -U ultralytics lap motmetrics filterpy kaggle
!yolo settings wandb=False > /dev/null

import torch, ultralytics
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__, "| GPUs:", torch.cuda.device_count())

for f in ["exps/example/mot/yolox_s_mot17_half.py", "exps/example/mot/yolox_tiny_mot17_half.py", "tools/track_dma.py"]:
    assert os.path.isfile(f), f"Thiếu {f}: hãy push code mới lên GitHub rồi chạy lại"
assert "det_pr" in open("yolox/evaluators/mot_evaluator.py").read(), \
    "mot_evaluator.py chưa có phần Precision/Recall: hãy push code mới lên GitHub"

/kaggle/working
/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
torch 2.11.0+cu128 | ultralytics 8.4.190 | GPUs: 2


## 3. Chuẩn bị MOT17 → `train_half` / `val_half` (COCO json cho YOLOX)

In [3]:
def find_mot17_train(root="/kaggle/input"):
    # os.walk nhưng không đi vào img1/ để chạy nhanh
    for dirpath, dirnames, _ in os.walk(root):
        dirnames[:] = [d for d in dirnames if d != "img1"]
        if "MOT17-02-FRCNN" in dirnames and os.path.isdir(os.path.join(dirpath, "MOT17-02-FRCNN", "gt")):
            return dirpath
    raise FileNotFoundError("Không tìm thấy MOT17 train/ (có MOT17-02-FRCNN/gt) trong /kaggle/input")

src_train = MOT17_TRAIN_SRC or find_mot17_train()
print("MOT17 train:", src_train)

MOT_ROOT = os.path.join(REPO_DIR, "datasets", "mot")
os.makedirs(os.path.join(MOT_ROOT, "test"), exist_ok=True)   # convert script cần thư mục test/
for seq in sorted(os.listdir(src_train)):
    if not seq.endswith("FRCNN"):
        continue
    s, d = os.path.join(src_train, seq), os.path.join(MOT_ROOT, "train", seq)
    os.makedirs(d, exist_ok=True)
    if not os.path.exists(os.path.join(d, "img1")):
        os.symlink(os.path.join(s, "img1"), os.path.join(d, "img1"))   # ảnh: symlink, không copy
    for sub in ["gt", "det"]:
        shutil.copytree(os.path.join(s, sub), os.path.join(d, sub), dirs_exist_ok=True)
    if os.path.isfile(os.path.join(s, "seqinfo.ini")):
        shutil.copy(os.path.join(s, "seqinfo.ini"), d)

if not os.path.isfile(os.path.join(MOT_ROOT, "annotations", "val_half.json")):
    !python tools/convert_mot17_to_coco.py > /dev/null
!ls {MOT_ROOT}/annotations

MOT17 train: /kaggle/input/datasets/hongchtrung/mot17-train/train
test.json  train_half.json  train.json	val_half.json


## 4. Chuyển cùng split sang định dạng YOLO (cho Ultralytics)

In [4]:
def coco_to_yolo(json_path, split):
    data = json.load(open(json_path))
    boxes = defaultdict(list)
    for a in data["annotations"]:
        if a["category_id"] == 1:                     # chỉ pedestrian, giống YOLOX/COCOeval
            boxes[a["image_id"]].append(a["bbox"])
    img_dir = os.path.join(YOLO_DATA_DIR, "images", split)
    lbl_dir = os.path.join(YOLO_DATA_DIR, "labels", split)
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)
    n_box = 0
    for im in data["images"]:
        name = im["file_name"].replace("/img1/", "_")             # MOT17-02-FRCNN_000001.jpg
        dst = os.path.join(img_dir, name)
        if not os.path.lexists(dst):
            os.symlink(os.path.join(MOT_ROOT, "train", im["file_name"]), dst)
        W, H = im["width"], im["height"]
        lines = []
        for x, y, w, h in boxes[im["id"]]:
            x1, y1, x2, y2 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)   # cắt box ra ngoài ảnh
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            lines.append(f"0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        n_box += len(lines)
        with open(os.path.join(lbl_dir, name[:-4] + ".txt"), "w") as f:
            f.write("\n".join(lines))
    print(f"{split}: {len(data['images'])} ảnh, {n_box} box")

coco_to_yolo(os.path.join(MOT_ROOT, "annotations", "train_half.json"), "train")
coco_to_yolo(os.path.join(MOT_ROOT, "annotations", "val_half.json"), "val")

DATA_YAML = os.path.join(YOLO_DATA_DIR, "mot17_half.yaml")
with open(DATA_YAML, "w") as f:
    f.write(f"path: {YOLO_DATA_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
print(open(DATA_YAML).read())

train: 2664 ảnh, 58407 box
val: 2652 ảnh, 53887 box
path: /kaggle/working/mot17_half_yolo
train: images/train
val: images/val
names:
  0: person



## 5. Khôi phục checkpoint cũ và bật sao lưu định kỳ

- **Khôi phục:** nếu dataset sao lưu đã được *Add Input*, chép các checkpoint về đúng chỗ cũ.
- **Sao lưu:** một luồng nền gom `last`/`best` của từng model và tạo version mới cho dataset sau mỗi `BACKUP_EVERY_MIN` phút. File nào đang được ghi dở (không load được) thì giữ bản cũ.

In [5]:
import threading

WORK = "/kaggle/working"
# Mỗi kích thước input lưu vào một thư mục riêng, để resume không trộn checkpoint khác size
YOLOX_IMGSZ_STR = f"{YOLOX_IMGSZ[0]}x{YOLOX_IMGSZ[1]}"
YOLOX_NAME = "yolox_s_mot17_half" + ("" if YOLOX_IMGSZ == (608, 1088) else f"_{YOLOX_IMGSZ_STR}")
YOLOX_OUT = os.path.join(REPO_DIR, "YOLOX_outputs", YOLOX_NAME)
YOLOX_LOG = os.path.join(RUNS_DIR, f"{YOLOX_NAME}_train.log")
STAGE = "/tmp/ckpt_backup"          # bản sao lưu đã kiểm tra, upload từ đây
SEP = "__"                          # đường dẫn được làm phẳng: runs/yolo11n/weights/last.pt -> runs__yolo11n__weights__last.pt
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(STAGE, exist_ok=True)

def files_to_backup():
    paths = [YOLOX_LOG, os.path.join(RUNS_DIR, "results_mot17_val_half.csv")]
    for w in ULTRA_MODELS:
        d = os.path.join(RUNS_DIR, os.path.splitext(w)[0])
        paths += [os.path.join(d, "weights", "last.pt"), os.path.join(d, "weights", "best.pt"),
                  os.path.join(d, "results.csv"), os.path.join(d, "args.yaml")]
    paths += [os.path.join(YOLOX_OUT, f) for f in ["latest_ckpt.pth.tar", "best_ckpt.pth.tar", "train_log.txt"]]
    paths += glob.glob(os.path.join(REPO_DIR, "YOLOX_outputs", "eval_half_*", "summary.json"))
    return [p for p in paths if os.path.isfile(p)]

def loads_ok(path):
    if not path.endswith((".pt", ".tar")):
        return True
    try:
        torch.load(path, map_location="cpu", weights_only=False)
        return True
    except Exception:
        return False

# --- Khôi phục ---------------------------------------------------------------
def find_backup_input(root="/kaggle/input"):
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in ("img1", "train", "test")]
        if "backup_manifest.json" in filenames:
            return dirpath
    return None

backup_in = find_backup_input()
if backup_in:
    n = 0
    for f in os.listdir(backup_in):
        if SEP not in f:
            continue
        dst = os.path.join(WORK, *f.split(SEP))
        shutil.copy(os.path.join(backup_in, f), os.path.join(STAGE, f))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(os.path.join(backup_in, f), dst)
            n += 1
    print(f"Khôi phục {n} file từ {backup_in}")
else:
    print("Không có bản sao lưu cũ: train từ đầu")

# --- Sao lưu -----------------------------------------------------------------
backup_lock = threading.Lock()
if BACKUP_ENABLED:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    except Exception as e:
        BACKUP_ENABLED = False
        print("⚠️ Không đọc được secrets KAGGLE_USERNAME / KAGGLE_KEY -> TẮT sao lưu:", e)

def backup(msg="auto"):
    if not BACKUP_ENABLED:
        return
    with backup_lock:
        changed = 0
        for p in files_to_backup():
            key = os.path.relpath(p, WORK).replace(os.sep, SEP)
            dst = os.path.join(STAGE, key)
            if os.path.exists(dst) and os.path.getmtime(dst) >= os.path.getmtime(p):
                continue
            tmp = dst + ".tmp"
            for _ in range(3):                      # file có thể đang được ghi dở
                shutil.copy(p, tmp)
                if loads_ok(tmp):
                    os.replace(tmp, dst); changed += 1
                    break
                time.sleep(20)
            if os.path.exists(tmp):
                os.remove(tmp)
        if not changed:
            return
        user = os.environ["KAGGLE_USERNAME"]
        json.dump({"title": BACKUP_SLUG, "id": f"{user}/{BACKUP_SLUG}", "licenses": [{"name": "CC0-1.0"}]},
                  open(os.path.join(STAGE, "dataset-metadata.json"), "w"))
        json.dump({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "msg": msg,
                   "files": sorted(f for f in os.listdir(STAGE) if SEP in f)},
                  open(os.path.join(STAGE, "backup_manifest.json"), "w"), indent=1)
        r = subprocess.run(["kaggle", "datasets", "version", "-p", STAGE, "-m", msg, "-q"],
                           capture_output=True, text=True, timeout=1800)
        if r.returncode != 0 or "error" in (r.stdout + r.stderr).lower():
            r = subprocess.run(["kaggle", "datasets", "create", "-p", STAGE, "-q"],
                               capture_output=True, text=True, timeout=1800)
        out = (r.stdout + r.stderr).strip().splitlines()
        print(f"[backup {time.strftime('%H:%M')}] {msg}: {changed} file mới |", out[-1] if out else "")

def backup_loop():
    while True:
        time.sleep(BACKUP_EVERY_MIN * 60)
        try:
            backup(f"auto {time.strftime('%H:%M')}")
        except Exception as e:
            print("[backup] lỗi:", e)

if BACKUP_ENABLED and not globals().get("_backup_thread_started"):
    threading.Thread(target=backup_loop, daemon=True).start()
    _backup_thread_started = True
    print(f"Sao lưu mỗi {BACKUP_EVERY_MIN} phút -> kaggle.com/datasets/{os.environ['KAGGLE_USERNAME']}/{BACKUP_SLUG}")

def ultra_status(name):
    # 'done' / 'resume' / 'new'
    d = os.path.join(RUNS_DIR, name)
    last = os.path.join(d, "weights", "last.pt")
    if not os.path.isfile(last):
        return "new"
    csv = os.path.join(d, "results.csv")
    n_epochs = max(sum(1 for _ in open(csv)) - 1, 0) if os.path.isfile(csv) else 0
    ck = torch.load(last, map_location="cpu", weights_only=False)
    return "done" if (ck.get("epoch") == -1 or n_epochs >= HP["epochs"]) else "resume"

def yolox_status():
    latest = os.path.join(YOLOX_OUT, "latest_ckpt.pth.tar")
    if not os.path.isfile(latest):
        return "new"
    ck = torch.load(latest, map_location="cpu", weights_only=False)
    return "done" if ck.get("start_epoch", 0) >= HP["epochs"] else "resume"

print({os.path.splitext(w)[0]: ultra_status(os.path.splitext(w)[0]) for w in ULTRA_MODELS}, "| yolox:", yolox_status())

Không có bản sao lưu cũ: train từ đầu
⚠️ Không đọc được secrets KAGGLE_USERNAME / KAGGLE_KEY -> TẮT sao lưu: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 141184997 and label KAGGLE_KEY.'], 'error': {'code': 5}, 'wasSuccessful': False}.
{'yolov10s': 'new', 'yolo11s': 'new', 'yolo12s': 'new', 'yolo26s': 'new'} | yolox: new


## 6. Train YOLOX-s (chạy nền)

Nếu có 2 GPU, YOLOX chạy nền trên GPU 1 trong khi Ultralytics dùng GPU 0. Nếu chỉ có 1 GPU, ô này chờ YOLOX train xong rồi mới sang bước sau.
Log ghi vào `runs/yolox_s_train.log`.

In [6]:
YOLOX_EXP = "exps/example/mot/yolox_s_mot17_half.py"
YOLOX_BEST = os.path.join(YOLOX_OUT, "best_ckpt.pth.tar")
yolox_proc = None
status = yolox_status()

if TRAIN_YOLOX and status == "done":
    print("YOLOX-s đã train xong: bỏ qua")
elif TRAIN_YOLOX:
    cmd = [sys.executable, "tools/train.py", "-f", YOLOX_EXP, "-expn", YOLOX_NAME,
           "-d", "1", "-b", str(HP["batch"]), "--fp16"]
    if status == "resume":
        # train tiếp: nạp model + optimizer + epoch + best_ap từ latest_ckpt
        cmd += ["--resume", "-c", os.path.join(YOLOX_OUT, "latest_ckpt.pth.tar")]
    else:
        os.makedirs("pretrained", exist_ok=True)
        coco_ckpt = "pretrained/yolox_s.pth"
        if not os.path.isfile(coco_ckpt):
            !wget -q -O {coco_ckpt} https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_s.pth
        # Lưu lại chỉ phần state_dict để torch.load(weights_only=True) của PyTorch mới đọc được
        sd = torch.load(coco_ckpt, map_location="cpu", weights_only=False)
        torch.save({"model": sd["model"] if "model" in sd else sd}, "pretrained/yolox_s_coco.pth")
        cmd += ["-c", "pretrained/yolox_s_coco.pth"]

    gpu = "1" if torch.cuda.device_count() > 1 else "0"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONPATH=REPO_DIR, YOLOX_IMGSZ=YOLOX_IMGSZ_STR)
    print(f"[{status}] GPU {gpu}, input {YOLOX_IMGSZ_STR}:", " ".join(cmd))
    yolox_proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env,
                                  stdout=open(YOLOX_LOG, "a"), stderr=subprocess.STDOUT)
    if gpu == "0":
        yolox_proc.wait()
        print("YOLOX exit code:", yolox_proc.returncode)
        backup("yolox finished")
        !tail -n 20 {YOLOX_LOG}

[new] GPU 1: /usr/bin/python3 tools/train.py -f exps/example/mot/yolox_tiny_mot17_half.py -expn yolox_tiny_mot17_half -d 1 -b 16 --fp16 -c pretrained/yolox_tiny_coco.pth


## 7. Train YOLOv10n / YOLO11n / YOLO12n

In [7]:
from ultralytics import YOLO

for w in ULTRA_MODELS:
    name = os.path.splitext(w)[0]
    status = ultra_status(name)
    print(f"\n===== {name}: {status} =====")
    if status == "done":
        continue
    if status == "resume":
        # train tiếp từ last.pt (giữ nguyên tham số, optimizer, epoch, best_fitness)
        YOLO(os.path.join(RUNS_DIR, name, "weights", "last.pt")).train(resume=True)
    else:
        YOLO(w).train(
            data=DATA_YAML, imgsz=ULTRA_IMGSZ, device=0, workers=WORKERS, seed=SEED, deterministic=True,
            project=RUNS_DIR, name=name, exist_ok=True, **HP,
        )
    backup(f"{name} finished")
    if yolox_proc is not None:
        print("YOLOX:", "đang chạy" if yolox_proc.poll() is None else f"xong (exit {yolox_proc.returncode})")


===== yolov10s: new =====
Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/mot17_half_yolo/mot17_half.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolov10s, nbs=6

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100       5.8G      1.512     0.9213     0.9989        236        640: 100% ━━━━━━━━━━━━ 167/167 3.9it/s 42.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 3.8it/s 21.8s
                   all       2652      53887      0.878      0.705      0.822      0.496

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      5.81G      1.441     0.8484     0.9746        281        640: 100% ━━━━━━━━━━━━ 167/167 3.9it/s 42.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 3.9it/s 21.4s
                   all       2652      53887      0.889      0.699      0.822      0.512

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      5.81G      1.418     0.8252     0.9652        226        640: 100% ━━━━━━━━━━━━ 167/167 3.9it/s 42.6s
                 Class     Images  Instances      B

## 8. Chờ YOLOX train xong

In [8]:
if yolox_proc is not None:
    while yolox_proc.poll() is None:
        time.sleep(300)
        print(time.strftime("%H:%M"), "|", subprocess.run(["tail", "-n", "1", YOLOX_LOG],
              capture_output=True, text=True).stdout.strip()[-200:])
    print("YOLOX exit code:", yolox_proc.returncode)
    backup("yolox finished")
!tail -n 30 {YOLOX_LOG}
print("best ckpt:", YOLOX_BEST, os.path.isfile(YOLOX_BEST))
# train.py dùng @logger.catch nên exit code vẫn là 0 khi bị lỗi -> kiểm tra log
if os.path.isfile(YOLOX_LOG) and "Traceback" in open(YOLOX_LOG, errors="ignore").read():
    print("❌ YOLOX train bị LỖI (xem traceback ở trên / trong", YOLOX_LOG, ")")
elif yolox_status() != "done":
    print("⚠️ YOLOX chưa train đủ", HP["epochs"], "epoch")

YOLOX exit code: 0
    return self._process_data(data, worker_id)
           │    │             │     └ 0
           │    │             └ <torch._utils.ExceptionWrapper object at 0x7d6d137d7230>
           │    └ <function _MultiProcessingDataLoaderIter._process_data at 0x7d6da33ce520>
           └ <torch.utils.data.dataloader._MultiProcessingDataLoaderIter object at 0x7d6d13631010>
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1586, in _process_data
    data.reraise()
    │    └ <function ExceptionWrapper.reraise at 0x7d6eb393c220>
    └ <torch._utils.ExceptionWrapper object at 0x7d6d137d7230>
  File "/usr/local/lib/python3.13/dist-packages/torch/_utils.py", line 785, in reraise
    raise exception
          └ IndexError('Caught IndexError in DataLoader worker process 0.\nOriginal Traceback (most recent call last):\n  File "/usr/loca...

IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  F

## 9. Đánh giá cả 4 model bằng cùng một pipeline (`tools/track_dma.py`)

Cả 4 model dùng chung COCOeval (Precision/Recall/mAP), chung ByteTrack và chung tập `val_half`, nên số liệu so sánh được với nhau.
Mỗi model dùng input mặc định của nó: Ultralytics `ULTRA_IMGSZ`, YOLOX-s `YOLOX_IMGSZ` (cùng size lúc train).

In [9]:
def run_track_dma(expn, extra, env_prefix=""):
    cmd = (f"{env_prefix}PYTHONPATH={REPO_DIR} python tools/track_dma.py -f {YOLOX_EXP} -expn {expn} -b 1 -d 1 --fp16 "
           f"--conf {EVAL_CONF} --nms {EVAL_NMS} {extra}")
    print(cmd)
    r = subprocess.run(cmd, shell=True, cwd=REPO_DIR, capture_output=True, text=True)
    open(os.path.join(RUNS_DIR, f"{expn}.log"), "w").write(r.stdout + r.stderr)
    print("  ->", "OK" if r.returncode == 0 else f"LỖI (xem runs/{expn}.log)")

EVALS = {}
if os.path.isfile(YOLOX_BEST):
    EVALS["YOLOX-s"] = f"eval_half_{YOLOX_NAME}"
    run_track_dma(EVALS["YOLOX-s"], f"-c {YOLOX_BEST} --fuse", env_prefix=f"YOLOX_IMGSZ={YOLOX_IMGSZ_STR} ")
for w in ULTRA_MODELS:
    name = os.path.splitext(w)[0]
    best = os.path.join(RUNS_DIR, name, "weights", "best.pt")
    if os.path.isfile(best):
        EVALS[name] = f"eval_half_{name}"
        run_track_dma(EVALS[name], f"--yolo-weights {best} --tsize {ULTRA_IMGSZ}")
backup("eval finished")

PYTHONPATH=/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker python tools/track_dma.py -f exps/example/mot/yolox_tiny_mot17_half.py -expn eval_half_yolov10s -b 1 -d 1 --fp16 --conf 0.001 --nms 0.7 --yolo-weights /kaggle/working/runs/yolov10s/weights/best.pt --tsize 640
  -> OK
PYTHONPATH=/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker python tools/track_dma.py -f exps/example/mot/yolox_tiny_mot17_half.py -expn eval_half_yolo11s -b 1 -d 1 --fp16 --conf 0.001 --nms 0.7 --yolo-weights /kaggle/working/runs/yolo11s/weights/best.pt --tsize 640
  -> OK
PYTHONPATH=/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker python tools/track_dma.py -f exps/example/mot/yolox_tiny_mot17_half.py -expn eval_half_yolo12s -b 1 -d 1 --fp16 --conf 0.001 --nms 0.7 --yolo-weights /kaggle/working/runs/yolo12s/weights/best.pt --tsize 640
  -> OK
PYTHONPATH=/kaggle/working/ByteTrack-DMA-LTC-Motion-Tracker python tools/track_dma.py -f exps/example/mot/yolox_tiny_mot17_half.py -expn eval_half_yolo26s -b 1 -d 1 --fp16 -

## 10. Bảng kết quả

In [10]:
import pandas as pd

rows = []
for model, expn in EVALS.items():
    p = os.path.join(REPO_DIR, "YOLOX_outputs", expn, "summary.json")
    if not os.path.isfile(p):
        continue
    s = json.load(open(p))
    det, pr = s["detection"], (s["detection"].get("PR50") or {}).get("best_f1", {})
    hota, spd, cx = s.get("hota") or {}, s.get("speed") or {}, s.get("detector_complexity") or {}
    rows.append({
        "Model": model, "Input": "x".join(map(str, s["test_size"])),
        "Params (M)": cx.get("params_M"), "FLOPs (G)": cx.get("flops_G"),
        "Precision": pr.get("precision"), "Recall": pr.get("recall"),
        "mAP50": det["AP50"], "mAP50-95": det["AP50_95"],
        "HOTA": hota.get("HOTA"), "MOTA": hota.get("MOTA"), "IDF1": hota.get("IDF1"),
        "FPS": spd.get("fps"),
    })
table = pd.DataFrame(rows).round(2)
table.to_csv(os.path.join(RUNS_DIR, "results_mot17_val_half.csv"), index=False)
backup("results table")
table

,Model,Input,Params (M),FLOPs (G),Precision,Recall,mAP50,mAP50-95,HOTA,MOTA,IDF1,FPS
0,yolov10s,640x640,7.22,21.41,83.09,63.0,69.62,39.26,50.62,51.29,60.10,29.71
1,yolo11s,640x640,9.41,21.30,83.23,63.0,70.46,39.53,50.94,51.89,61.27,30.07
2,yolo12s,640x640,9.23,21.23,81.71,65.0,69.83,39.14,51.97,52.77,62.71,28.08
3,yolo26s,640x640,9.47,20.52,82.23,67.0,72.63,41.37,51.16,52.83,61.76,30.43


### (Tuỳ chọn) Số liệu theo cách Ultralytics tự tính

Đây là cách bài báo báo cáo. Chỉ áp dụng cho 3 model Ultralytics nên **không dùng để so với YOLOX**, chỉ để đối chiếu với bài báo.

In [11]:
ultra_rows = []
for w in ULTRA_MODELS:
    name = os.path.splitext(w)[0]
    best = os.path.join(RUNS_DIR, name, "weights", "best.pt")
    if not os.path.isfile(best):
        continue
    m = YOLO(best).val(data=DATA_YAML, imgsz=ULTRA_IMGSZ, batch=16, conf=EVAL_CONF, iou=EVAL_NMS,
                       device=0, plots=False, verbose=False, project=RUNS_DIR, name=f"val_{name}", exist_ok=True)
    ultra_rows.append({"Model": name, "Precision": m.box.mp * 100, "Recall": m.box.mr * 100,
                       "mAP50": m.box.map50 * 100, "mAP50-95": m.box.map * 100})
pd.DataFrame(ultra_rows).round(2)

Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv10s summary (fused): 103 layers, 7,218,387 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 208.2±139.6 MB/s, size: 117.8 KB)
val: Scanning /kaggle/working/mot17_half_yolo/labels/val.cache... 2652 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2652/2652 252.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 166/166 6.4it/s 25.9s
                   all       2652      53887      0.889      0.699      0.822      0.512
Speed: 0.6ms preprocess, 5.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Ultralytics 8.4.190 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 224.0±93.1 MB/s, size: 163.2 KB)
val: Scanning /kaggle/working/mot17_half_yolo/

,Model,Precision,Recall,mAP50,mAP50-95
0,yolov10s,88.90,69.90,82.23,51.23
1,yolo11s,89.91,70.22,82.18,52.05
2,yolo12s,89.84,71.37,80.80,50.34
3,yolo26s,90.59,72.67,83.92,53.14


## 11. Đóng gói kết quả

Tải `/kaggle/working/results_mot17_half.zip` ở tab *Output* sau khi *Save Version*.

In [12]:
pack = "/kaggle/working/results_pack"
shutil.rmtree(pack, ignore_errors=True)
os.makedirs(pack)
for w in ULTRA_MODELS:
    name = os.path.splitext(w)[0]
    for f in ["weights/best.pt", "results.csv", "args.yaml"]:
        src = os.path.join(RUNS_DIR, name, f)
        if os.path.isfile(src):
            os.makedirs(os.path.join(pack, name), exist_ok=True)
            shutil.copy(src, os.path.join(pack, name, os.path.basename(f)))
if os.path.isfile(YOLOX_BEST):
    os.makedirs(os.path.join(pack, YOLOX_NAME), exist_ok=True)
    shutil.copy(YOLOX_BEST, os.path.join(pack, YOLOX_NAME))
    shutil.copy(YOLOX_LOG, os.path.join(pack, YOLOX_NAME))
for expn in EVALS.values():
    d = os.path.join(REPO_DIR, "YOLOX_outputs", expn)
    for f in ["summary.json", "val_log.txt"]:
        if os.path.isfile(os.path.join(d, f)):
            os.makedirs(os.path.join(pack, "eval", expn), exist_ok=True)
            shutil.copy(os.path.join(d, f), os.path.join(pack, "eval", expn))
for f in glob.glob(os.path.join(RUNS_DIR, "*.csv")) + glob.glob(os.path.join(RUNS_DIR, "*.log")):
    shutil.copy(f, pack)
shutil.make_archive("/kaggle/working/results_mot17_half", "zip", pack)
!ls -la /kaggle/working/results_mot17_half.zip

-rw-r--r-- 1 root root 68950362 Oct 27 22:17 /kaggle/working/results_mot17_half.zip
